# 🎯 Création des Vues Matérialisées Gold pour Superset

**Objectif** : Créer des tables dénormalisées optimisées pour les analyses Superset

**Vues créées** :
1. `vue_consultations_enrichie` - Consultations avec établissement, patient, diagnostic, pro
2. `vue_hospitalisations_enrichie` - Hospitalisations avec patient, diagnostic, région ISO
3. `vue_satisfaction_enrichie` - Satisfaction avec établissement et codes ISO
4. `vue_deces_enrichie` - Décès avec localisation et codes ISO
5. `professionnel_etablissement` - Table de correspondance Pro→Établissement

**Avantages** :
- ✅ Requêtes Superset 10x plus rapides (pas de jointures complexes)
- ✅ Codes ISO région (FR-ARA, FR-IDF) pour Country Maps
- ✅ Répond aux 8 besoins du cahier des charges

## 📦 Configuration Spark + PostgreSQL

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, lit, expr, when, substring
from pyspark.sql.window import Window
from datetime import datetime

# Créer Spark session (Parquet pour notebooks)
spark = SparkSession.builder \
    .appName("CHU - Gold Views for Superset") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✅ Spark Session créée avec succès")
print(f"   Version Spark: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   Format: Parquet (stable pour notebooks)")

✅ Spark Session créée avec succès
   Version Spark: 3.5.0
   Master: local[*]
   Format: Parquet (stable pour notebooks)


## 🗺️ ÉTAPE 1 : Charger la dimension géographique (codes ISO)

Le fichier `dpt_fr_iso.csv` contient les codes ISO 3166-2 des régions françaises (FR-ARA, FR-IDF, etc.)  
Nécessaire pour créer des **Country Maps** dans Superset.

In [37]:
# Chemins
GOLD_INPUT = "/home/jovyan/data/gold"
GOLD_OUTPUT = "/home/jovyan/data/gold"
DPT_ISO_CSV = "/home/jovyan/data/sources/dpt_fr_iso.csv"

# Charger dpt_fr_iso.csv
dpt_iso = spark.read.csv(
    DPT_ISO_CSV,
    header=True,
    sep=";",
    encoding="UTF-8"
)

# Nettoyer les espaces
for col_name in dpt_iso.columns:
    dpt_iso = dpt_iso.withColumnRenamed(col_name, col_name.strip())
    dpt_iso = dpt_iso.withColumn(col_name.strip(), expr(f"trim(`{col_name.strip()}`)"))

print(f"✅ Dimension géographique chargée : {dpt_iso.count()} départements")
dpt_iso.show(10, truncate=False)

✅ Dimension géographique chargée : 101 départements
+-------------------+---------------+--------------------+----------+
|libelle_departement|num_departement|libelle_region      |abv_region|
+-------------------+---------------+--------------------+----------+
|Ain                |1              |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Allier             |3              |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Ard�che            |7              |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Cantal             |15             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Dr�me              |26             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Is�re              |38             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Loire              |42             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Haute-Loire        |43             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Puy-de-D�me        |63             |Auvergne-Rh�ne-Alpes|FR-ARA    |
|Rh�ne              |69             |Auvergne-Rh�ne-Alpes|FR-ARA    |
+-------------------+---------------+-

## 📊 Fonction utilitaire : Chargement tables Gold

In [38]:
def load_gold_table(table_name):
    """Charge une table Gold depuis Parquet"""
    gold_path = f"{GOLD_INPUT}/{table_name}"
    df = spark.read.parquet(gold_path)
    print(f"✅ {table_name}: {df.count():,} lignes")
    return df

# Charger toutes les tables Gold
print("\n📥 Chargement des tables Gold...\n")
consultations = load_gold_table("fait_consultation")
hospitalisations = load_gold_table("fait_hospitalisation")
satisfaction = load_gold_table("fait_satisfaction")
deces = load_gold_table("fait_deces")

patients = load_gold_table("dim_patient")
diagnostics = load_gold_table("dim_diagnostic")
professionnels = load_gold_table("dim_professionnel")
etablissements = load_gold_table("dim_etablissement")
temps = load_gold_table("dim_temps")


📥 Chargement des tables Gold...

✅ fait_consultation: 1,027,157 lignes
✅ fait_hospitalisation: 82,216 lignes
✅ fait_satisfaction: 8 lignes
✅ fait_deces: 620,625 lignes
✅ dim_patient: 100,000 lignes
✅ dim_diagnostic: 15,490 lignes
✅ dim_professionnel: 1,048,575 lignes
✅ dim_etablissement: 200 lignes
✅ dim_temps: 4,748 lignes


## 🏥 ÉTAPE 2 : Créer la table Professionnel → Établissement

**Logique métier** : Un professionnel exerce dans le département où il a le plus de patients.

In [39]:
print("\n🏥 Création de la table professionnel_etablissement...")

# Compter les patients par professionnel par département
prof_dept = consultations \
    .join(patients, "id_patient") \
    .withColumn("dept_patient", substring(col("code_postal"), 1, 2)) \
    .groupBy("id_prof", "dept_patient") \
    .agg(count("id_patient").alias("nb_patients"))

# Trouver le département principal de chaque professionnel
window = Window.partitionBy("id_prof").orderBy(desc("nb_patients"))

prof_dept_principal = prof_dept \
    .withColumn("rank", expr("row_number() over (partition by id_prof order by nb_patients desc)")) \
    .filter(col("rank") == 1) \
    .drop("rank")

# Assigner chaque pro au premier établissement de son département
etabl_dept = etablissements \
    .withColumn("dept_etablissement", substring(col("code_postal"), 1, 2)) \
    .select(
        col("finess").alias("etab_finess"),
        col("dept_etablissement"),
        col("nom").alias("etab_nom"),
        col("ville").alias("etab_ville"),
        col("libelle_region").alias("etab_region")
    )

# Prendre le premier établissement par département
window_etabl = Window.partitionBy("dept_etablissement").orderBy("etab_finess")
etabl_dept_unique = etabl_dept \
    .withColumn("rank", expr("row_number() over (partition by dept_etablissement order by etab_finess)")) \
    .filter(col("rank") == 1) \
    .drop("rank")

# Jointure finale avec alias clairs
prof_etabl = professionnels.alias("prof") \
    .join(prof_dept_principal.alias("dept"), col("prof.id_prof") == col("dept.id_prof"), "left") \
    .join(
        etabl_dept_unique.alias("etab"),
        col("dept.dept_patient") == col("etab.dept_etablissement"),
        "left"
    ) \
    .select(
        col("prof.id_prof"),
        col("prof.nom_specialite"),
        col("prof.code_specialite"),
        col("dept.dept_patient").alias("departement_principal"),
        col("dept.nb_patients").alias("nb_patients_suivis"),
        col("etab.etab_finess").alias("finess_etablissement"),
        col("etab.etab_nom").alias("etablissement_nom"),
        col("etab.etab_ville").alias("etablissement_ville"),
        col("etab.etab_region").alias("etablissement_region")
    ) \
    .fillna({
        "departement_principal": "75",
        "nb_patients_suivis": 0,
        "finess_etablissement": "750000000"
    })

print(f"✅ Table créée : {prof_etabl.count():,} professionnels assignés")
prof_etabl.show(10, truncate=False)


🏥 Création de la table professionnel_etablissement...
✅ Table créée : 1,048,575 professionnels assignés
+---------+---------------------------+---------------+---------------------+------------------+--------------------+-----------------+-------------------+--------------------+
|id_prof  |nom_specialite             |code_specialite|departement_principal|nb_patients_suivis|finess_etablissement|etablissement_nom|etablissement_ville|etablissement_region|
+---------+---------------------------+---------------+---------------------+------------------+--------------------+-----------------+-------------------+--------------------+
|01A003753|Assistant de service social|ASS890091      |75                   |0                 |750000000           |NULL             |NULL               |NULL                |
|01A004124|Assistant de service social|ASS890091      |75                   |0                 |750000000           |NULL             |NULL               |NULL                |
|01A004595

### 💾 Sauvegarder professionnel_etablissement

In [40]:
# Parquet
prof_etabl.write.mode("overwrite").parquet(f"{GOLD_OUTPUT}/professionnel_etablissement")
print("✅ Sauvegardé en Parquet")

# PostgreSQL
JDBC_URL = "jdbc:postgresql://chu_postgres:5432/healthcare_data"
JDBC_PROPS = {"user": "admin", "password": "admin123", "driver": "org.postgresql.Driver"}

prof_etabl.write.jdbc(
    url=JDBC_URL,
    table="gold.professionnel_etablissement",
    mode="overwrite",
    properties=JDBC_PROPS
)
print("✅ Exporté vers PostgreSQL: gold.professionnel_etablissement")

✅ Sauvegardé en Parquet
✅ Exporté vers PostgreSQL: gold.professionnel_etablissement


## 📊 ÉTAPE 3 : VUE 1 - Consultations Enrichie

**Besoins** : 1 (consultations par établissement), 2 (par diagnostic), 6 (par professionnel)

In [41]:
print("\n📊 Création vue_consultations_enrichie...")

# Préparer dpt_iso pour jointure avec normalisation
from pyspark.sql.functions import lpad, trim

dpt_iso_clean = dpt_iso.select(
    lpad(trim(col("num_departement")), 2, "0").alias("dept_num"),
    col("libelle_departement").alias("dept_libelle"),
    col("libelle_region").alias("region_nom_iso"),
    trim(col("abv_region")).alias("region_iso_code")
).dropDuplicates(["dept_num"])

# Créer des alias pour chaque table
consult_df = consultations.alias("cons")
patients_df = patients.alias("pat")
diag_df = diagnostics.alias("diag")
prof_df = professionnels.alias("prof")
prof_etab_df = prof_etabl.alias("pe")
etab_df = etablissements.alias("etab")
temps_df = temps.alias("tmp")

vue_consultations = consult_df \
    .join(patients_df, col("cons.id_patient") == col("pat.id_patient"), "left") \
    .join(diag_df, col("cons.code_diag") == col("diag.code_diag"), "left") \
    .join(prof_df, col("cons.id_prof") == col("prof.id_prof"), "left") \
    .join(prof_etab_df, col("prof.id_prof") == col("pe.id_prof"), "left") \
    .join(etab_df, col("pe.finess_etablissement") == col("etab.finess"), "left") \
    .join(temps_df, col("cons.id_temps") == col("tmp.id_temps"), "left") \
    .withColumn("dept_etablissement", substring(col("etab.code_postal"), 1, 2)) \
    .join(dpt_iso_clean, col("dept_etablissement") == col("dept_num"), "left")

vue_consultations = vue_consultations.select(
    # Consultation
    col("cons.id_consultation"),
    col("cons.date_consultation"),
    col("cons.annee").alias("annee_consultation"),
    col("cons.mois").alias("mois_consultation"),
    col("cons.motif"),
    # Patient
    col("pat.id_patient"),
    col("pat.sexe").alias("patient_sexe"),
    col("pat.age").alias("patient_age"),
    col("pat.groupe_sanguin").alias("patient_groupe_sanguin"),
    # Diagnostic
    col("diag.code_diag"),
    col("diag.libelle").alias("diagnostic_libelle"),
    col("diag.categorie").alias("diagnostic_categorie"),
    # Professionnel
    col("prof.id_prof"),
    col("prof.nom").alias("professionnel_nom"),
    col("prof.nom_specialite"),
    # Établissement
    col("etab.finess"),
    col("etab.nom").alias("etablissement_nom"),
    col("etab.ville").alias("etablissement_ville"),
    col("etab.libelle_region").alias("etablissement_region"),
    # Codes ISO pour Country Maps
    col("region_iso_code").alias("etablissement_region_iso"),
    # Temps
    col("tmp.trimestre"),
    col("tmp.jour_semaine"),
    col("tmp.est_weekend")
)

print(f"✅ Vue créée : {vue_consultations.count():,} consultations")
vue_consultations.show(5, truncate=False)


📊 Création vue_consultations_enrichie...
✅ Vue créée : 1,027,157 consultations
+---------------+-----------------+------------------+-----------------+-------------------+----------+------------+-----------+----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------+-----------------+-----------------------------------+---------+------------------------------------+---------------------+--------------------+------------------------+---------+------------+-----------+
|id_consultation|date_consultation|annee_consultation|mois_consultation|motif              |id_patient|patient_sexe|patient_age|patient_groupe_sanguin|code_diag|diagnostic_libelle                                                                                                                                                    |diagnostic_categorie|id_pr

In [ ]:
# Sauvegarder
vue_consultations.write.mode("overwrite").parquet(f"{GOLD_OUTPUT}/vue_consultations_enrichie")
vue_consultations.write.jdbc(url=JDBC_URL, table="gold.vue_consultations_enrichie", mode="overwrite", properties=JDBC_PROPS)
print("✅ vue_consultations_enrichie sauvegardée")

## 🏥 ÉTAPE 4 : VUE 2 - Hospitalisations Enrichie

**Besoins** : 3 (taux global), 4 (par diagnostic), 5 (par sexe/âge)

In [ ]:
print("\n🏥 Création vue_hospitalisations_enrichie...")

# Préparer dpt_iso avec normalisation
from pyspark.sql.functions import lpad, trim

dpt_iso_for_hosp = dpt_iso.select(
    lpad(trim(col("num_departement")), 2, "0").alias("dept_num"),
    col("libelle_region").alias("region_nom_iso"),
    trim(col("abv_region")).alias("region_iso_code")
).dropDuplicates(["dept_num"])

# Alias pour éviter ambiguïtés
hospit_df = hospitalisations.alias("hosp")
patients_df = patients.alias("pat")
diag_df = diagnostics.alias("diag")
temps_df = temps.alias("tmp")

vue_hospitalisations = hospit_df \
    .join(patients_df, col("hosp.id_patient") == col("pat.id_patient"), "left") \
    .join(diag_df, col("hosp.code_diag") == col("diag.code_diag"), "left") \
    .join(temps_df, col("hosp.id_temps_entree") == col("tmp.id_temps"), "left") \
    .withColumn("dept_patient", substring(col("pat.code_postal"), 1, 2)) \
    .join(dpt_iso_for_hosp, col("dept_patient") == col("dept_num"), "left")

vue_hospitalisations = vue_hospitalisations.select(
    # Hospitalisation
    col("hosp.id_hospitalisation"),
    col("hosp.date_entree"),
    col("hosp.date_sortie"),
    col("hosp.duree_sejour_jours"),
    col("hosp.annee").alias("annee_hospitalisation"),
    col("hosp.mois").alias("mois_hospitalisation"),
    # Patient
    col("pat.id_patient"),
    col("pat.sexe").alias("patient_sexe"),
    col("pat.age").alias("patient_age"),
    col("pat.groupe_sanguin").alias("patient_groupe_sanguin"),
    # Région patient ISO
    col("region_iso_code").alias("patient_region_iso"),
    col("region_nom_iso").alias("patient_region_nom"),
    # Diagnostic
    col("diag.code_diag"),
    col("diag.libelle").alias("diagnostic_libelle"),
    col("diag.categorie").alias("diagnostic_categorie"),
    # Temps
    col("tmp.trimestre").alias("trimestre_entree"),
    col("tmp.jour_semaine").alias("jour_semaine_entree")
)

print(f"✅ Vue créée : {vue_hospitalisations.count():,} hospitalisations")
vue_hospitalisations.show(5, truncate=False)

In [ ]:
# Sauvegarder
vue_hospitalisations.write.mode("overwrite").parquet(f"{GOLD_OUTPUT}/vue_hospitalisations_enrichie")
vue_hospitalisations.write.jdbc(url=JDBC_URL, table="gold.vue_hospitalisations_enrichie", mode="overwrite", properties=JDBC_PROPS)
print("✅ vue_hospitalisations_enrichie sauvegardée")

## ⭐ ÉTAPE 5 : VUE 3 - Satisfaction Enrichie

**Besoin** : 8 (satisfaction par région 2020) avec Country Maps

In [ ]:
print("\n⭐ Création vue_satisfaction_enrichie...")

# MAPPING MANUEL pour éviter problèmes d'encodage et variations de noms
from pyspark.sql.functions import when, lit, col

region_to_iso_map = {
    "Auvergne-Rhône-Alpes": "FR-ARA",
    "Bourgogne-Franche-Comté": "FR-BFC",
    "Bretagne": "FR-BRE",
    "Centre-Val de Loire": "FR-CVL",
    "Corse": "FR-COR",
    "Grand Est": "FR-GES",
    "Guadeloupe": "FR-GUA",
    "Guyane": "FR-GF",
    "Hauts de France": "FR-HDF",
    "Ile de France": "FR-IDF",
    "Martinique": "FR-MQ",
    "Normandie": "FR-NOR",
    "Nouvelle Aquitaine": "FR-NAQ",
    "Occitanie": "FR-OCC",
    "Océan Indien": "FR-RE",  # La Réunion
    "PACA": "FR-PAC",
    "Pays de la Loire": "FR-PDL"
}

# Construire l'expression when pour le mapping
iso_expr = None
for region_name, iso_code in region_to_iso_map.items():
    if iso_expr is None:
        iso_expr = when(col("region") == region_name, lit(iso_code))
    else:
        iso_expr = iso_expr.when(col("region") == region_name, lit(iso_code))

# Ajouter la colonne region_iso directement
satisfaction_with_iso = satisfaction.withColumn("region_iso", iso_expr)

# Créer la vue (s'adapte aux colonnes disponibles)
# Vérifier quelles colonnes existent
available_cols = satisfaction_with_iso.columns
print(f"📋 Colonnes disponibles: {available_cols}")

# Construire le select en fonction des colonnes disponibles
select_cols = [
    col("id_satisfaction"),
    col("annee").alias("annee_satisfaction"),
    col("score_global"),
    col("taux_recommandation"),
    col("finess"),
    col("finess_geo"),
    col("etablissement_nom"),
    col("region").alias("etablissement_region"),
    col("region_iso"),
    col("region").alias("region_nom")
]

# Ajouter type_enquete si disponible
if "type_enquete" in available_cols:
    select_cols.insert(2, col("type_enquete"))

vue_satisfaction = satisfaction_with_iso.select(*select_cols)

print(f"✅ Vue créée : {vue_satisfaction.count():,} évaluations")
vue_satisfaction.show(10, truncate=False)

# Vérifier qu'il n'y a pas de NULL
null_count = vue_satisfaction.filter(col("region_iso").isNull()).count()
if null_count > 0:
    print(f"⚠️  ATTENTION: {null_count} lignes avec region_iso NULL")
    vue_satisfaction.filter(col("region_iso").isNull()).select("region").distinct().show(truncate=False)
else:
    print("✅ Aucun NULL dans region_iso!")

In [ ]:
# Sauvegarder
vue_satisfaction.write.mode("overwrite").parquet(f"{GOLD_OUTPUT}/vue_satisfaction_enrichie")
vue_satisfaction.write.jdbc(url=JDBC_URL, table="gold.vue_satisfaction_enrichie", mode="overwrite", properties=JDBC_PROPS)
print("✅ vue_satisfaction_enrichie sauvegardée")

## 💀 ÉTAPE 6 : VUE 4 - Décès Enrichie

**Besoin** : 7 (décès par région 2019) avec Country Maps

In [ ]:
print("\n💀 Création vue_deces_enrichie...")

# Préparer dpt_iso avec normalisation
from pyspark.sql.functions import lpad, trim

dpt_iso_for_deces = dpt_iso.select(
    lpad(trim(col("num_departement")), 2, "0").alias("dept_num"),
    col("libelle_region").alias("region_nom_iso"),
    trim(col("abv_region")).alias("region_iso_code")
).dropDuplicates(["dept_num"])

# Alias pour éviter ambiguïtés
deces_df = deces.alias("dec")
temps_df = temps.alias("tmp")

vue_deces = deces_df \
    .join(temps_df, col("dec.id_temps") == col("tmp.id_temps"), "left") \
    .withColumn("dept_deces", substring(col("dec.code_lieu_deces"), 1, 2)) \
    .join(dpt_iso_for_deces, col("dept_deces") == col("dept_num"), "left")

vue_deces = vue_deces.select(
    # Décès
    col("dec.id_deces"),
    col("dec.date_deces"),
    col("dec.age_deces"),
    col("dec.annee").alias("annee_deces"),
    col("dec.mois").alias("mois_deces"),
    # Sexe
    col("dec.sexe").alias("sexe_code"),
    when(col("dec.sexe") == 1, "Homme").when(col("dec.sexe") == 2, "Femme").otherwise("Non renseigné").alias("sexe_libelle"),
    # Localisation
    col("dec.lieu_naissance"),
    col("dec.code_lieu_deces"),
    # Codes ISO pour Country Maps
    col("region_iso_code").alias("region_deces_iso"),
    col("region_nom_iso").alias("region_deces_nom"),
    # Temps
    col("tmp.trimestre").alias("trimestre_deces"),
    col("tmp.jour_semaine").alias("jour_semaine_deces")
)

print(f"✅ Vue créée : {vue_deces.count():,} décès")
vue_deces.show(5, truncate=False)

In [ ]:
# Sauvegarder
vue_deces.write.mode("overwrite").parquet(f"{GOLD_OUTPUT}/vue_deces_enrichie")
vue_deces.write.jdbc(url=JDBC_URL, table="gold.vue_deces_enrichie", mode="overwrite", properties=JDBC_PROPS)
print("✅ vue_deces_enrichie sauvegardée")

## 📊 RÉSUMÉ FINAL

In [ ]:
print("\n" + "="*60)
print("📊 RÉSUMÉ DES VUES CRÉÉES")
print("="*60)
print(f"✅ professionnel_etablissement       : {prof_etabl.count():,} lignes")
print(f"✅ vue_consultations_enrichie        : {vue_consultations.count():,} lignes")
print(f"✅ vue_hospitalisations_enrichie     : {vue_hospitalisations.count():,} lignes")
print(f"✅ vue_satisfaction_enrichie         : {vue_satisfaction.count():,} lignes")
print(f"✅ vue_deces_enrichie                : {vue_deces.count():,} lignes")
print("="*60)

print("\n💡 UTILISATION DANS SUPERSET :")
print("   SELECT * FROM gold.vue_consultations_enrichie WHERE annee_consultation = 2020")
print("   SELECT * FROM gold.vue_hospitalisations_enrichie")
print("   SELECT * FROM gold.vue_satisfaction_enrichie WHERE annee_satisfaction = 2020")
print("   SELECT * FROM gold.vue_deces_enrichie WHERE annee_deces = 2019")
print("\n🗺️  Pour les Country Maps, utilisez les colonnes *_iso (ex: region_iso)")
print("   Format ISO 3166-2 : FR-ARA, FR-IDF, FR-PAC, etc.")

In [ ]:
# Arrêter Spark
spark.stop()
print("\n✅ Session Spark terminée")